# Chapter 9.8: Online Serving vs Offline Batch Inference

LLM workloads fall into two categories: latency-sensitive online serving and
throughput-optimized offline batch processing. This notebook explores both modes
and hybrid approaches.

## Learning Objectives
- Understand the tradeoffs between online and offline inference
- Use vLLM's offline LLM class for batch processing
- Implement SGLang batch processing
- Build async batch processing with queues
- Design hybrid systems with priority queues

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hideak1/vllm_study/blob/main/notebooks/part9/chapter_9.8_online_vs_offline.ipynb)
[![Download Notebook](https://img.shields.io/badge/Download-Notebook-blue)](https://raw.githubusercontent.com/hideak1/vllm_study/main/notebooks/part9/chapter_9.8_online_vs_offline.ipynb)

**How to do the exercises:**
1. **Google Colab (Recommended)**: Click the "Open in Colab" badge above — you get your own copy with free GPU.
2. **Local Jupyter**: Clone the repo, run `./start.sh`, then open notebooks at `http://localhost:8888`.
3. **Exercises**: Look for cells marked with 🏋️ **Exercise** or **Assignment**. Fill in the `TODO` sections and run the cell to check your work.

> **Tip**: In Colab, the notebook is automatically copied to your Drive — your changes are saved there.

In [ ]:
import json
import time
import random
import asyncio
import statistics
from dataclasses import dataclass, field
from typing import Optional
from collections import deque
import heapq

print("Setup complete.")

## 1. Online vs Offline: Key Differences

In [ ]:
# Side-by-side comparison

COMPARISON = {
    "Characteristic": [
        "Latency requirement",
        "Throughput priority",
        "Streaming support",
        "Batch size",
        "Request arrival",
        "SLO type",
        "Optimal config",
        "Cost optimization",
        "Failure handling",
        "Typical use cases",
    ],
    "Online Serving": [
        "Critical (< 2-5s TTFT)",
        "Secondary to latency",
        "Required for UX",
        "Dynamic (continuous batching)",
        "Unpredictable, bursty",
        "P99 latency < X seconds",
        "Lower batch, prefix caching",
        "Auto-scale, reserved instances",
        "Immediate retry, fallback",
        "Chatbots, code assist, search",
    ],
    "Offline Batch": [
        "Not critical (hours OK)",
        "Maximize tokens/second/$",
        "Not needed",
        "Large static batches",
        "Controlled, planned",
        "Total time < X hours",
        "Max batch, high GPU util",
        "Spot instances, schedule off-peak",
        "Checkpoint + resume",
        "Eval, dataset gen, embeddings",
    ],
}

print(f"{'Characteristic':<25} | {'Online Serving':<35} | {'Offline Batch'}")
print("-" * 100)
for i in range(len(COMPARISON["Characteristic"])):
    print(f"{COMPARISON['Characteristic'][i]:<25} | "
          f"{COMPARISON['Online Serving'][i]:<35} | "
          f"{COMPARISON['Offline Batch'][i]}")

In [ ]:
# Configuration differences

ONLINE_CONFIG = {
    "description": "Optimized for latency-sensitive online serving",
    "vllm_args": {
        "--max-num-seqs": 64,           # Moderate batch
        "--max-model-len": 4096,
        "--gpu-memory-utilization": 0.85,  # Leave headroom
        "--enable-prefix-caching": True,
        "--enable-chunked-prefill": True,
        "--max-num-batched-tokens": 4096,
        "--disable-log-requests": True,   # Reduce overhead
    },
}

OFFLINE_CONFIG = {
    "description": "Optimized for maximum throughput batch processing",
    "vllm_args": {
        "--max-num-seqs": 256,           # Large batch
        "--max-model-len": 8192,
        "--gpu-memory-utilization": 0.95,  # Max utilization
        "--enable-prefix-caching": True,
        "--enable-chunked-prefill": True,
        "--max-num-batched-tokens": 16384,
        "--swap-space": 16,              # More swap for large batches
    },
}

print("Online Serving Configuration:")
for k, v in ONLINE_CONFIG["vllm_args"].items():
    print(f"  {k} {v}")

print("\nOffline Batch Configuration:")
for k, v in OFFLINE_CONFIG["vllm_args"].items():
    print(f"  {k} {v}")

## 2. Demo: vLLM Offline Batch with LLM Class

In [ ]:
# vLLM offline batch inference code
# (Shows the API; requires vLLM installed with GPU)

VLLM_OFFLINE_CODE = '''
from vllm import LLM, SamplingParams
import json
import time

# Initialize the model (downloads and loads to GPU)
llm = LLM(
    model="meta-llama/Llama-3.1-8B-Instruct",
    tensor_parallel_size=1,
    max_model_len=4096,
    gpu_memory_utilization=0.95,  # Max utilization for batch
    enable_prefix_caching=True,
)

# Sampling parameters
sampling_params = SamplingParams(
    temperature=0.7,
    top_p=0.9,
    max_tokens=256,
)

# Prepare batch of prompts
prompts = [
    "Explain quantum computing in simple terms.",
    "Write a Python function to sort a list.",
    "What are the benefits of renewable energy?",
    "Describe the process of photosynthesis.",
    # ... add thousands of prompts
]

# Process in batches
start = time.time()
outputs = llm.generate(prompts, sampling_params)
elapsed = time.time() - start

# Collect results
results = []
total_tokens = 0
for output in outputs:
    generated_text = output.outputs[0].text
    num_tokens = len(output.outputs[0].token_ids)
    total_tokens += num_tokens
    results.append({
        "prompt": output.prompt[:100],
        "output": generated_text[:200],
        "tokens": num_tokens,
    })

print(f"Processed {len(prompts)} prompts in {elapsed:.1f}s")
print(f"Total tokens: {total_tokens}")
print(f"Throughput: {total_tokens/elapsed:.0f} tokens/s")

# Save results
with open("batch_results.jsonl", "w") as f:
    for r in results:
        f.write(json.dumps(r) + "\\n")
'''

print("vLLM Offline Batch Inference Code:")
print("=" * 50)
print(VLLM_OFFLINE_CODE)

In [ ]:
# Chat template batch processing

VLLM_CHAT_BATCH = '''
from vllm import LLM, SamplingParams

llm = LLM(
    model="meta-llama/Llama-3.1-8B-Instruct",
    gpu_memory_utilization=0.95,
)

# Batch of chat conversations
conversations = [
    [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Summarize the key points of machine learning."},
    ],
    [
        {"role": "system", "content": "You are a code reviewer."},
        {"role": "user", "content": "Review this code: def fib(n): return fib(n-1)+fib(n-2)"},
    ],
    # ... more conversations
]

sampling_params = SamplingParams(temperature=0.7, max_tokens=512)

# Use chat method for proper template formatting
outputs = llm.chat(conversations, sampling_params)

for conv, output in zip(conversations, outputs):
    user_msg = conv[-1]["content"]
    response = output.outputs[0].text
    print(f"User: {user_msg[:50]}...")
    print(f"Assistant: {response[:100]}...\\n")
'''

print("Chat-formatted Batch Processing:")
print(VLLM_CHAT_BATCH)

## 3. Demo: SGLang Batch Processing

In [ ]:
SGLANG_BATCH_CODE = '''
import sglang as sgl

# Method 1: Using the offline engine
from sglang import Engine

engine = Engine(
    model_path="meta-llama/Llama-3.1-8B-Instruct",
    tp_size=1,
)

prompts = [
    "What is machine learning?",
    "Explain neural networks.",
    "What is deep learning?",
]

# Generate all at once
outputs = engine.generate(
    prompts,
    sampling_params={"temperature": 0.7, "max_new_tokens": 256}
)

for prompt, output in zip(prompts, outputs):
    print(f"Prompt: {prompt}")
    print(f"Output: {output['text'][:100]}...\\n")

engine.shutdown()


# Method 2: HTTP batch using the server
import requests
import concurrent.futures

BASE_URL = "http://localhost:30000"

def send_request(prompt):
    response = requests.post(
        f"{BASE_URL}/v1/chat/completions",
        json={
            "model": "default",
            "messages": [{"role": "user", "content": prompt}],
            "max_tokens": 256,
            "temperature": 0.7,
        }
    )
    return response.json()

# Send batch concurrently
with concurrent.futures.ThreadPoolExecutor(max_workers=32) as executor:
    futures = {executor.submit(send_request, p): p for p in prompts}
    for future in concurrent.futures.as_completed(futures):
        prompt = futures[future]
        result = future.result()
        print(f"Done: {prompt[:30]}...")
'''

print("SGLang Batch Processing:")
print(SGLANG_BATCH_CODE)

## 4. Demo: Async Batch Processing with Queues

In [ ]:
@dataclass
class InferenceRequest:
    """A request in the processing queue."""
    request_id: str
    prompt: str
    max_tokens: int = 256
    priority: int = 0       # Higher = more urgent (0=batch, 10=interactive)
    submitted_at: float = 0.0
    started_at: float = 0.0
    completed_at: float = 0.0
    result: str = ""
    status: str = "pending"  # pending, processing, completed, failed
    
    def __lt__(self, other):
        # Higher priority first, then earlier submission
        if self.priority != other.priority:
            return self.priority > other.priority
        return self.submitted_at < other.submitted_at


class AsyncBatchProcessor:
    """Process inference requests asynchronously with batching."""
    
    def __init__(self, max_batch_size: int = 32, batch_timeout_ms: float = 100,
                 simulated_throughput: float = 2000):
        self.max_batch_size = max_batch_size
        self.batch_timeout_ms = batch_timeout_ms
        self.throughput = simulated_throughput  # tokens/s
        
        self.queue: list = []  # Priority queue
        self.results: dict[str, InferenceRequest] = {}
        self.batches_processed = 0
        self.total_tokens = 0
    
    def submit(self, request: InferenceRequest):
        """Submit a request to the queue."""
        request.submitted_at = time.time()
        request.status = "pending"
        heapq.heappush(self.queue, request)
        self.results[request.request_id] = request
    
    def _form_batch(self) -> list[InferenceRequest]:
        """Form a batch from the queue."""
        batch = []
        while self.queue and len(batch) < self.max_batch_size:
            req = heapq.heappop(self.queue)
            batch.append(req)
        return batch
    
    def process_batch(self) -> list[InferenceRequest]:
        """Process one batch of requests."""
        batch = self._form_batch()
        if not batch:
            return []
        
        now = time.time()
        total_tokens = sum(r.max_tokens for r in batch)
        process_time = total_tokens / self.throughput  # Simulated
        
        for req in batch:
            req.started_at = now
            req.status = "processing"
        
        # Simulate processing
        time.sleep(min(process_time, 0.1))  # Cap sleep for demo
        
        for req in batch:
            req.completed_at = time.time()
            req.result = f"[Generated {req.max_tokens} tokens]"
            req.status = "completed"
            self.total_tokens += req.max_tokens
        
        self.batches_processed += 1
        return batch
    
    def get_status(self, request_id: str) -> dict:
        """Check the status of a request."""
        req = self.results.get(request_id)
        if not req:
            return {"error": "not found"}
        return {
            "request_id": req.request_id,
            "status": req.status,
            "queue_time_ms": (req.started_at - req.submitted_at) * 1000 if req.started_at else 0,
            "process_time_ms": (req.completed_at - req.started_at) * 1000 if req.completed_at else 0,
            "total_time_ms": (req.completed_at - req.submitted_at) * 1000 if req.completed_at else 0,
        }
    
    def stats(self) -> dict:
        return {
            "queue_depth": len(self.queue),
            "batches_processed": self.batches_processed,
            "total_tokens": self.total_tokens,
            "total_requests": len(self.results),
            "completed": sum(1 for r in self.results.values() if r.status == "completed"),
        }


# Demo: process mixed online and batch requests
processor = AsyncBatchProcessor(max_batch_size=16)

# Submit batch requests (low priority)
for i in range(50):
    processor.submit(InferenceRequest(
        request_id=f"batch-{i}",
        prompt=f"Batch prompt {i}",
        max_tokens=256,
        priority=0,
    ))

# Submit interactive requests (high priority)
for i in range(5):
    processor.submit(InferenceRequest(
        request_id=f"interactive-{i}",
        prompt=f"Interactive prompt {i}",
        max_tokens=128,
        priority=10,
    ))

print(f"Queue depth: {len(processor.queue)}")

# Process batches
while processor.queue:
    batch = processor.process_batch()
    priorities = [r.priority for r in batch]
    ids = [r.request_id for r in batch]
    print(f"  Batch {processor.batches_processed}: {len(batch)} requests, "
          f"priorities: {set(priorities)}, first: {ids[0]}")

# Check interactive requests were processed first
print("\nInteractive request completion times:")
for i in range(5):
    status = processor.get_status(f"interactive-{i}")
    print(f"  {status['request_id']}: total={status['total_time_ms']:.1f}ms")

print(f"\nStats: {processor.stats()}")

## 5. Optimal Configurations for Each Mode

In [ ]:
def simulate_mode_comparison(
    num_requests: int = 1000,
    avg_input_tokens: int = 300,
    avg_output_tokens: int = 200,
    seed: int = 42,
) -> dict:
    """Simulate online vs offline inference and compare metrics."""
    rng = random.Random(seed)
    
    results = {}
    
    for mode in ["online", "offline"]:
        if mode == "online":
            max_batch = 32
            gpu_util = 0.75
            base_ttft_ms = 100
            base_itl_ms = 15
        else:
            max_batch = 256
            gpu_util = 0.95
            base_ttft_ms = 500  # Higher but doesn't matter
            base_itl_ms = 8     # Lower due to better batching
        
        latencies = []
        ttfts = []
        total_tokens = 0
        start = time.time()
        
        for i in range(num_requests):
            inp = rng.randint(100, 500)
            out = rng.randint(50, 400)
            
            batch_factor = min(i % max_batch + 1, max_batch) / max_batch
            
            ttft = base_ttft_ms * (1 + batch_factor * 2) * rng.uniform(0.8, 1.2)
            itl = base_itl_ms * (1 + batch_factor * 0.5) * rng.uniform(0.8, 1.2)
            e2e = ttft + out * itl
            
            latencies.append(e2e)
            ttfts.append(ttft)
            total_tokens += out
        
        elapsed = time.time() - start
        # Adjust total time based on mode
        if mode == "online":
            total_time = sum(latencies) / 1000 / max_batch  # Concurrent
        else:
            total_time = sum(latencies) / 1000 / max_batch * 0.5  # Better packing
        
        latencies.sort()
        ttfts.sort()
        
        results[mode] = {
            "throughput_tok_s": round(total_tokens / total_time, 0),
            "throughput_req_s": round(num_requests / total_time, 1),
            "e2e_p50_ms": round(latencies[len(latencies)//2], 0),
            "e2e_p99_ms": round(latencies[int(len(latencies)*0.99)], 0),
            "ttft_p50_ms": round(ttfts[len(ttfts)//2], 0),
            "ttft_p99_ms": round(ttfts[int(len(ttfts)*0.99)], 0),
            "gpu_utilization": gpu_util,
            "max_batch_size": max_batch,
        }
    
    return results


comparison = simulate_mode_comparison(1000)

print(f"{'Metric':<25} {'Online':>15} {'Offline':>15} {'Difference':>15}")
print("=" * 75)

for metric in comparison["online"]:
    online_val = comparison["online"][metric]
    offline_val = comparison["offline"][metric]
    
    if isinstance(online_val, float):
        diff = offline_val / online_val if online_val != 0 else 0
        diff_str = f"{diff:.2f}x"
        print(f"{metric:<25} {online_val:>15.1f} {offline_val:>15.1f} {diff_str:>15}")
    else:
        diff = offline_val / online_val if online_val != 0 else 0
        diff_str = f"{diff:.2f}x"
        print(f"{metric:<25} {online_val:>15} {offline_val:>15} {diff_str:>15}")

## 6. Hybrid: Online Serving with Batch Queue

In [ ]:
class HybridServingSystem:
    """Serve both interactive and batch requests on the same GPU."""
    
    def __init__(self, gpu_capacity_tok_s: float = 3000):
        self.capacity = gpu_capacity_tok_s
        self.online_queue = deque()     # FIFO for interactive
        self.batch_queue = deque()      # FIFO for batch
        self.online_reservation = 0.7   # Reserve 70% for online
        self.processed = {"online": [], "batch": []}
    
    def submit_online(self, request_id: str, tokens: int):
        self.online_queue.append({
            "id": request_id, "tokens": tokens, 
            "type": "online", "submitted": time.time()
        })
    
    def submit_batch(self, request_id: str, tokens: int):
        self.batch_queue.append({
            "id": request_id, "tokens": tokens,
            "type": "batch", "submitted": time.time()
        })
    
    def process_tick(self) -> dict:
        """Process one scheduling tick (simulates 100ms)."""
        available_tokens = self.capacity * 0.1  # 100ms worth
        
        # Phase 1: Process online requests first (reserved capacity)
        online_budget = available_tokens * self.online_reservation
        online_processed = 0
        while self.online_queue and online_budget > 0:
            req = self.online_queue[0]
            if req["tokens"] <= online_budget:
                self.online_queue.popleft()
                online_budget -= req["tokens"]
                online_processed += 1
                req["completed"] = time.time()
                req["latency_ms"] = (req["completed"] - req["submitted"]) * 1000
                self.processed["online"].append(req)
            else:
                break
        
        # Phase 2: Use remaining capacity for batch
        batch_budget = available_tokens - (available_tokens * self.online_reservation - online_budget)
        # Also use leftover online budget
        batch_budget += online_budget
        
        batch_processed = 0
        while self.batch_queue and batch_budget > 0:
            req = self.batch_queue[0]
            if req["tokens"] <= batch_budget:
                self.batch_queue.popleft()
                batch_budget -= req["tokens"]
                batch_processed += 1
                req["completed"] = time.time()
                req["latency_ms"] = (req["completed"] - req["submitted"]) * 1000
                self.processed["batch"].append(req)
            else:
                break
        
        return {
            "online_processed": online_processed,
            "batch_processed": batch_processed,
            "online_queue": len(self.online_queue),
            "batch_queue": len(self.batch_queue),
        }
    
    def summary(self) -> dict:
        online_latencies = [r["latency_ms"] for r in self.processed["online"]]
        batch_latencies = [r["latency_ms"] for r in self.processed["batch"]]
        
        return {
            "online": {
                "completed": len(online_latencies),
                "avg_latency_ms": round(statistics.mean(online_latencies), 1) if online_latencies else 0,
                "p99_latency_ms": round(sorted(online_latencies)[int(len(online_latencies)*0.99)], 1) if online_latencies else 0,
            },
            "batch": {
                "completed": len(batch_latencies),
                "avg_latency_ms": round(statistics.mean(batch_latencies), 1) if batch_latencies else 0,
            },
            "remaining_online": len(self.online_queue),
            "remaining_batch": len(self.batch_queue),
        }


# Simulate hybrid serving
system = HybridServingSystem(gpu_capacity_tok_s=3000)
rng = random.Random(42)

# Submit mixed workload
for i in range(100):
    system.submit_batch(f"batch-{i}", rng.randint(100, 400))

for i in range(20):
    system.submit_online(f"online-{i}", rng.randint(50, 200))

# Process
for tick in range(50):
    # Add sporadic online requests
    if rng.random() < 0.3:
        system.submit_online(f"online-late-{tick}", rng.randint(50, 200))
    
    result = system.process_tick()

summary = system.summary()
print("Hybrid Serving Results:")
print(f"  Online: {summary['online']['completed']} completed, "
      f"avg latency: {summary['online']['avg_latency_ms']:.1f}ms, "
      f"P99: {summary['online']['p99_latency_ms']:.1f}ms")
print(f"  Batch:  {summary['batch']['completed']} completed, "
      f"avg latency: {summary['batch']['avg_latency_ms']:.1f}ms")
print(f"  Remaining: {summary['remaining_online']} online, {summary['remaining_batch']} batch")

---
## Hands-on Assignments

### Assignment 1: Build an Offline Batch Inference Pipeline

In [ ]:
# ASSIGNMENT 1: Build a complete offline batch pipeline

class OfflineBatchPipeline:
    """Complete offline batch inference pipeline.
    
    TODO: Implement a pipeline that:
    1. Reads prompts from a JSONL file
    2. Groups into optimal batch sizes
    3. Processes with progress tracking
    4. Handles failures with retry logic
    5. Writes results to output JSONL
    6. Supports checkpointing for resume
    """
    
    def __init__(self, batch_size: int = 64, max_retries: int = 3,
                 checkpoint_interval: int = 100):
        self.batch_size = batch_size
        self.max_retries = max_retries
        self.checkpoint_interval = checkpoint_interval
        self.processed: list[dict] = []
        self.failed: list[dict] = []
        self.checkpoint_file = "checkpoint.json"
    
    def load_prompts(self, data: list[dict]) -> list[dict]:
        """Load and validate prompts."""
        # TODO: Validate each prompt has required fields
        return data
    
    def create_batches(self, prompts: list[dict]) -> list[list[dict]]:
        """Group prompts into batches, optionally sorting by length."""
        # TODO: Sort by estimated token count for better batching efficiency
        return []
    
    def process_batch(self, batch: list[dict]) -> list[dict]:
        """Process a single batch (simulated)."""
        # TODO: Simulate inference with random failures
        return []
    
    def run(self, prompts: list[dict]) -> dict:
        """Run the complete pipeline."""
        # TODO: Implement the full pipeline with checkpointing
        return {}

pipeline = OfflineBatchPipeline()
print("Implement the OfflineBatchPipeline methods!")

In [ ]:
# ASSIGNMENT 1 - SOLUTION

class OfflineBatchPipelineSolution:
    def __init__(self, batch_size=64, max_retries=3, checkpoint_interval=100):
        self.batch_size = batch_size
        self.max_retries = max_retries
        self.checkpoint_interval = checkpoint_interval
        self.processed = []
        self.failed = []
        self.rng = random.Random(42)
    
    def load_prompts(self, data):
        validated = []
        for item in data:
            if "prompt" not in item:
                continue
            item.setdefault("max_tokens", 256)
            item.setdefault("id", f"req-{len(validated)}")
            validated.append(item)
        return validated
    
    def create_batches(self, prompts):
        # Sort by prompt length for better batching
        sorted_prompts = sorted(prompts, key=lambda p: len(p["prompt"]))
        batches = []
        for i in range(0, len(sorted_prompts), self.batch_size):
            batches.append(sorted_prompts[i:i+self.batch_size])
        return batches
    
    def process_batch(self, batch):
        results = []
        for item in batch:
            # Simulate with 5% failure rate
            if self.rng.random() < 0.05:
                results.append({**item, "status": "failed", "error": "simulated error"})
            else:
                output_tokens = self.rng.randint(50, item["max_tokens"])
                results.append({
                    **item,
                    "status": "completed",
                    "output": f"[Generated {output_tokens} tokens]",
                    "output_tokens": output_tokens,
                })
        return results
    
    def run(self, prompts_data):
        prompts = self.load_prompts(prompts_data)
        batches = self.create_batches(prompts)
        
        total = len(prompts)
        completed = 0
        failed = 0
        total_tokens = 0
        start = time.time()
        
        for batch_idx, batch in enumerate(batches):
            results = self.process_batch(batch)
            
            # Retry failures
            for attempt in range(self.max_retries):
                failures = [r for r in results if r["status"] == "failed"]
                if not failures:
                    break
                retry_results = self.process_batch(failures)
                # Update results
                for rr in retry_results:
                    for i, r in enumerate(results):
                        if r["id"] == rr["id"]:
                            results[i] = rr
            
            for r in results:
                if r["status"] == "completed":
                    completed += 1
                    total_tokens += r.get("output_tokens", 0)
                    self.processed.append(r)
                else:
                    failed += 1
                    self.failed.append(r)
            
            # Progress
            if (batch_idx + 1) % 5 == 0 or batch_idx == len(batches) - 1:
                pct = (completed + failed) / total * 100
                print(f"  Progress: {pct:.0f}% ({completed} ok, {failed} failed)")
        
        elapsed = time.time() - start
        return {
            "total": total,
            "completed": completed,
            "failed": failed,
            "total_tokens": total_tokens,
            "elapsed_s": round(elapsed, 2),
            "success_rate": round(completed / total, 3) if total > 0 else 0,
        }


# Test the pipeline
test_data = [{"prompt": f"Question {i}: explain concept {i}", "max_tokens": 200} for i in range(500)]
pipeline = OfflineBatchPipelineSolution(batch_size=32)

print("Running offline batch pipeline:")
result = pipeline.run(test_data)
print(f"\nFinal Results: {json.dumps(result, indent=2)}")

### Assignment 2: Compare Online vs Offline Throughput

In [ ]:
# ASSIGNMENT 2: Throughput comparison under different conditions

def compare_throughput(
    request_rates: list[float] = [1, 5, 10, 20, 50, 100],
    gpu_capacity_tok_s: float = 3000,
    max_online_batch: int = 32,
    max_offline_batch: int = 256,
) -> dict:
    """Compare online vs offline throughput at different request rates.
    
    TODO: For each request rate, calculate:
    1. Online: effective throughput considering batching and latency constraints
    2. Offline: effective throughput with optimal batching
    3. Utilization: what % of GPU capacity is used
    4. Latency: P50 and P99 for each mode
    """
    rng = random.Random(42)
    results = {}
    
    for rate in request_rates:
        avg_tokens = 200
        
        # Online: limited by batch size and must maintain latency
        online_batch = min(max_online_batch, max(1, int(rate * 0.5)))  # Dynamic batching
        online_throughput = min(gpu_capacity_tok_s * 0.75, rate * avg_tokens)
        online_util = online_throughput / gpu_capacity_tok_s
        online_latency = avg_tokens / (gpu_capacity_tok_s / online_batch) * 1000
        
        # Offline: can use full GPU capacity
        offline_throughput = gpu_capacity_tok_s * 0.95
        offline_util = 0.95
        offline_latency = avg_tokens / (gpu_capacity_tok_s / max_offline_batch) * 1000
        
        results[rate] = {
            "online_throughput": round(online_throughput, 0),
            "offline_throughput": round(offline_throughput, 0),
            "online_util": round(online_util, 2),
            "offline_util": offline_util,
            "online_latency_ms": round(online_latency, 0),
            "offline_latency_ms": round(offline_latency, 0),
            "throughput_ratio": round(offline_throughput / max(1, online_throughput), 2),
        }
    
    return results

results = compare_throughput()

print(f"{'Rate':>6} {'Online tok/s':>13} {'Offline tok/s':>14} {'Ratio':>7} "
      f"{'Online Util':>12} {'Online Lat':>11}")
print("=" * 70)
for rate, m in results.items():
    print(f"{rate:>5}/s {m['online_throughput']:>12.0f} {m['offline_throughput']:>14.0f} "
          f"{m['throughput_ratio']:>6.1f}x {m['online_util']:>10.0%} {m['online_latency_ms']:>9.0f}ms")

### Assignment 3: Implement Priority Queue with Batch Fallback

In [ ]:
# ASSIGNMENT 3: Priority queue with batch fallback

class PriorityBatchSystem:
    """Serve interactive requests with priority, falling back to batch when idle.
    
    TODO: Implement a system that:
    1. Interactive requests get immediate processing (P0)
    2. Premium batch gets next priority (P1)
    3. Standard batch gets remaining capacity (P2)
    4. When GPU is idle (<30% util), aggressively process batch
    5. When GPU is busy (>80% util), only process interactive
    6. Track SLO compliance for each tier
    """
    
    def __init__(self, gpu_capacity: float = 3000):
        self.capacity = gpu_capacity
        self.queues = {0: deque(), 1: deque(), 2: deque()}  # priority -> queue
        self.slo_targets = {0: 2000, 1: 30000, 2: 300000}  # ms
        self.processed = {0: [], 1: [], 2: []}
        self.current_util = 0.0
    
    def submit(self, request_id: str, tokens: int, priority: int):
        self.queues[priority].append({
            "id": request_id, "tokens": tokens, 
            "priority": priority, "submitted": time.time()
        })
    
    def process_tick(self, tick_duration_s: float = 0.1):
        """Process one scheduling tick."""
        budget = self.capacity * tick_duration_s
        used = 0
        
        # TODO: Process queues by priority
        # TODO: Adjust batch processing based on utilization
        
        for priority in [0, 1, 2]:
            # Adaptive: skip low priority when busy
            if priority > 0 and self.current_util > 0.8:
                continue
            
            while self.queues[priority] and used < budget:
                req = self.queues[priority][0]
                if req["tokens"] + used <= budget:
                    self.queues[priority].popleft()
                    used += req["tokens"]
                    req["completed"] = time.time()
                    req["latency_ms"] = (req["completed"] - req["submitted"]) * 1000
                    self.processed[priority].append(req)
                else:
                    break
        
        self.current_util = used / budget if budget > 0 else 0
    
    def slo_report(self):
        """Report SLO compliance per tier."""
        print("\nSLO Compliance Report:")
        print(f"{'Tier':<12} {'Completed':>10} {'SLO Target':>12} {'P99 Latency':>12} {'Compliant':>10}")
        print("=" * 60)
        
        tier_names = {0: "Interactive", 1: "Premium", 2: "Standard"}
        for priority in [0, 1, 2]:
            processed = self.processed[priority]
            if not processed:
                continue
            latencies = sorted([r["latency_ms"] for r in processed])
            p99 = latencies[int(len(latencies) * 0.99)]
            target = self.slo_targets[priority]
            compliant = p99 <= target
            
            print(f"{tier_names[priority]:<12} {len(processed):>10} "
                  f"{target:>10}ms {p99:>10.0f}ms {'YES' if compliant else 'NO':>10}")


# Simulate mixed workload
system = PriorityBatchSystem(gpu_capacity=3000)
rng = random.Random(42)

# Submit workload
for i in range(200):
    system.submit(f"batch-{i}", rng.randint(100, 400), priority=2)  # Standard batch

for i in range(50):
    system.submit(f"premium-{i}", rng.randint(100, 300), priority=1)  # Premium

# Run simulation with sporadic interactive requests
for tick in range(100):
    if rng.random() < 0.3:
        system.submit(f"interactive-{tick}", rng.randint(50, 200), priority=0)
    system.process_tick()

system.slo_report()

total = sum(len(v) for v in system.processed.values())
remaining = sum(len(q) for q in system.queues.values())
print(f"\nTotal processed: {total}, Remaining in queue: {remaining}")

## Summary

Key takeaways:

1. **Online serving** prioritizes latency: use moderate batch sizes, prefix caching, streaming
2. **Offline batch** maximizes throughput: use large batches, high GPU utilization, no latency constraint
3. **Offline can be 2-5x more cost-efficient** than online due to better GPU utilization
4. **vLLM's LLM class** is ideal for offline batch; the API server is for online
5. **Hybrid systems** serve interactive requests with priority and fill idle time with batch
6. **Priority queues** ensure interactive SLOs while maximizing batch throughput
7. **Checkpointing** is essential for offline batch to survive interruptions (especially with spot instances)
8. **Sort by length** before batching offline requests for better padding efficiency